# 02 — Attach political party

Read the v1 parquet built by `src/preprocessing.py`, match `page_name` and `bylines` against the 2022 AEC candidate list, add a `political_party` column (`PartyAb` or null), and write a new v2 parquet partitioned by party.

## 1. Spark session

In [ ]:
from pyspark.sql import SparkSession
import pandas as pd
import re

spark = SparkSession.builder \
    .appName('FB_API_party_match') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

Master: yarn
Spark version: 3.5.0


## 2. Paths

In [7]:
IN_PATH        = '/user/s3348393/main/preprocessing/v1/parquet'
OUT_PATH       = '/user/s3348393/main/preprocessing/v2/parquet'
CANDIDATES_CSV = '../data/2022_election_candidates.csv'

## 3. Load v1 parquet

In [4]:
df = spark.read.parquet(IN_PATH)
print('Rows:', df.count())
df.printSchema()

Rows: 3128023
root
 |-- id: string (nullable = true)
 |-- page_id: string (nullable = true)
 |-- page_name: string (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- ad_creation_date: date (nullable = true)
 |-- ad_delivery_start_date: date (nullable = true)
 |-- ad_delivery_stop_date: date (nullable = true)
 |-- creative_bodies: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creative_link_captions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creative_link_descs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- creative_link_titles: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- currency: string (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- publisher_platforms: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- demographic_distribution: array (nullable = true)
 |

## 4. Load 2022 candidates

Columns: `DivisionNm, PartyAb, PartyNm, Surname, GivenNm`. Names are uppercase; surnames can be multi-word (`ABDUL RAZAK`), given-name field can be empty.

In [10]:
candidates = pd.read_csv(CANDIDATES_CSV, header=0)
print('Candidates:', len(candidates))
candidates.head()

Candidates: 1203


,DivisionNm,PartyAb,PartyNm,Surname,GivenNm
0,Goldstein,ALP,Australian Labor Party,ABBOTT,Martyn
1,Calwell,GVIC,The Greens,ABBOUD,Natalie
2,Tangney,GRN,The Greens (WA),ABDUL RAZAK,Adam
3,La Trobe,LDP,Liberal Democrats,ABELMAN,Michael
4,La Trobe,ALP,Australian Labor Party,ABHIMANYU KUMAR,NaN


## 5. Tokenise `page_name` and `bylines`

Use Spark MLlib `RegexTokenizer` (splits on `\W+`, lowercases) + `StopWordsRemover` with English defaults extended by political honorifics. Fields are tokenised independently — we won't concatenate them before matching, to avoid a given name in `page_name` crossing with a surname in `bylines` to spuriously match a candidate.

`coalesce(col, lit(''))` because `RegexTokenizer` errors on null inputs.

In [11]:
from pyspark.sql.functions import coalesce, col, lit
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

df = df.withColumn('page_name_safe', coalesce(col('page_name'), lit(''))) \
       .withColumn('bylines_safe',   coalesce(col('bylines'),   lit('')))

honorifics = ['mp', 'hon', 'dr', 'mr', 'mrs', 'ms', 'sen', 'senator', 'rt', 'authorised', 'by', 'for']
stopwords = StopWordsRemover.loadDefaultStopWords('english') + honorifics

for src, tok_col, term_col in [
    ('page_name_safe', 'page_name_tokens', 'page_name_terms'),
    ('bylines_safe',   'bylines_tokens',   'bylines_terms'),
]:
    df = RegexTokenizer(inputCol=src, outputCol=tok_col, pattern=r'\W+', toLowercase=True).transform(df)
    df = StopWordsRemover(inputCol=tok_col, outputCol=term_col, stopWords=stopwords).transform(df)

df.select('page_name', 'page_name_terms', 'bylines', 'bylines_terms').show(5, truncate=80)

+--------------+---------------+--------------+--------------+
|     page_name|page_name_terms|       bylines| bylines_terms|
+--------------+---------------+--------------+--------------+
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
|Thrive by Five| [thrive, five]|Thrive By Five|[thrive, five]|
+--------------+---------------+--------------+--------------+
only showing top 5 rows



## 6. Build candidate match index

For each candidate, the *required tokens* are the first given-name token plus all surname tokens — e.g. Adam ABDUL RAZAK → `{adam, abdul, razak}`. Token-set subset matching is order-insensitive, so 'Adam Abdul Razak' and 'Abdul Razak, Adam' both match.

Index by the first surname token to keep matching fast: at lookup time we only check candidates whose key token appears in the ad's token list, rather than scanning all ~1,500 candidates per ad.

In [ ]:
def split_tokens(s):
    if not isinstance(s, str):
        return []
    return [t for t in re.split(r'\W+', s.lower()) if t]

candidate_list = []
for _, row in candidates.iterrows():
    surname_toks = split_tokens(row['Surname'])
    if not surname_toks:
        continue
    given_toks = split_tokens(row['GivenNm'])
    required = frozenset(surname_toks + given_toks[:1])
    party = row['PartyAb']
    if not isinstance(party, str):
        continue
    candidate_list.append((required, party))

print('Candidates indexed:', len(candidate_list))

Candidates indexed: 1203


## 7. Assign `political_party`

For each ad: look up `page_name_terms` and `bylines_terms` independently. A candidate matches a field if its required-token set is a subset of that field's tokens. Collect the set of matched `PartyAb`s across both fields — if it resolves to exactly one party, that's the label; otherwise null (conservative: ambiguous matches stay unlabelled).

In [15]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(StringType())
def match_party(page_terms, bylines_terms):
    page_set    = set(page_terms or [])
    bylines_set = set(bylines_terms or [])

    parties = set()
    for required, party in candidate_list:
        if required.issubset(page_set) or required.issubset(bylines_set):
            parties.add(party)

    if len(parties) == 1:
        return next(iter(parties))
    return None

df = df.withColumn('political_party', match_party('page_name_terms', 'bylines_terms'))

## 8. Sanity checks

In [16]:
df.groupBy('political_party').count().orderBy(col('count').desc()).show(50, truncate=False)

+---------------+-------+
|political_party|count  |
+---------------+-------+
|NULL           |2647852|
|ALP            |204812 |
|LP             |130105 |
|IND            |60868  |
|LNP            |15573  |
|GRN            |14646  |
|NP             |14123  |
|ON             |9103   |
|GVIC           |8444   |
|UAPP           |7010   |
|LDP            |4837   |
|JLN            |3069   |
|TNL            |2713   |
|CLP            |895    |
|XEN            |891    |
|CYA            |414    |
|AUVA           |370    |
|SOPA           |365    |
|VNS            |360    |
|KAP            |321    |
|IMO            |243    |
|TLOC           |220    |
|SAL            |211    |
|NaN            |178    |
|AJP            |172    |
|DPDA           |167    |
|GAP            |45     |
|REAS           |9      |
|SPP            |7      |
+---------------+-------+



In [20]:
# spot-check: a few matched rows per top party
from pyspark.sql.functions import desc

top_parties = [r['political_party'] for r in
               df.filter(col('political_party').isNotNull())
                 .groupBy('political_party').count()
                 .orderBy(desc('count')).limit(5).collect()]

for p in top_parties:
    print(f'\n=== {p} ===')
    df.filter(col('political_party') == p).select('page_name', 'bylines', 'political_party').distinct().show(10, truncate=60)


=== ALP ===


+--------------------------------------+--------------------------------+---------------+
|                             page_name|                         bylines|political_party|
+--------------------------------------+--------------------------------+---------------+
|                     Kristina Keneally|               Kristina Keneally|            ALP|
|                       Alicia Payne MP|                 Alicia Payne MP|            ALP|
|      Tabatha Young - Labor for Bonner|Tabatha Young - labor for Bonner|            ALP|
|                            Mary Doyle|       Mary Judith Jacinta Doyle|            ALP|
|                        Anika Wells MP|                  Anika Wells MP|            ALP|
|                     Richard Marles MP|               Richard Marles MP|            ALP|
|Bronwen English - WA Labor for Forrest|                        WA Labor|            ALP|
|Andrew Charlton - Labor for Parramatta|                 Andrew Charlton|            ALP|
|        A

+-------------------------------------+-----------------------------------------------+---------------+
|                            page_name|                                        bylines|political_party|
+-------------------------------------+-----------------------------------------------+---------------+
|                         Zoe McKenzie|Liberal Party of Australia (Victorian Division)|             LP|
|      Shawn Lock - Liberal for Spence|                      South Australian Liberals|             LP|
|                            Ken Wyatt|                                      Ken Wyatt|             LP|
|                       Jason Falinski|                                 Jason Falinski|             LP|
|                      Angus Taylor MP|                                Angus Taylor MP|             LP|
|                        Richard Welch|Liberal Party of Australia (Victorian Division)|             LP|
|                     Celia Hammond MP|                         

+-----------------------------------------------------+--------------------------------------+---------------+
|                                            page_name|                               bylines|political_party|
+-----------------------------------------------------+--------------------------------------+---------------+
|                                        Zali Steggall|                         Zali Steggall|            IND|
|                                         Jack Dempsey|              Jack Dempsey for Hinkler|            IND|
|               North Sydney's Kylea Tink for Canberra|                            Kylea Tink|            IND|
|                 Liz Habermann - Independent for Grey|                         Liz Habermann|            IND|
|Matt Sharpham - Independent Candidate for New England|                Matthew Peter Sharpham|            IND|
|                            Rob Priestly for Nicholls|            Rob Priestly for Nicholls |            IND|
|

+-----------------------------------------------+------------------------------------+---------------+
|                                      page_name|                             bylines|political_party|
+-----------------------------------------------+------------------------------------+---------------+
|                                Michelle Landry|                     Michelle Landry|            LNP|
|                                  Angie Bell MP|                       Angie Bell MP|            LNP|
|            Colin Boyce MP - Member for Callide| Colin Boyce MP - Member for Callide|            LNP|
|                                  Ross Vasta MP|                       Ross Vasta MP|            LNP|
| Andrew Wallace - LNP Federal Member for Fisher|Liberal National Party of Queensland|            LNP|
|                               Stuart Robert MP|                    Stuart Robert MP|            LNP|
|                   Colin Boyce  - LNP for Flynn|                        

+--------------------------------------------------+--------------------------+---------------+
|                                         page_name|                   bylines|political_party|
+--------------------------------------------------+--------------------------+---------------+
|        Natasa Sojic - Greens Candidate for Fenner|                ACT Greens|            GRN|
|Kristyn Glanville - Greens candidate for Curl Curl|    The Greens NSW - Manly|            GRN|
|          Elizabeth Watson-Brown - Greens for Ryan|     The Australian Greens|            GRN|
|             Rachael Jacobs - Greens for Grayndler|            The Greens NSW|            GRN|
|                               Eli Davern - Greens|            The Greens NSW|            GRN|
|                Taylor Vandijk - Greens for Barton|            The Greens NSW|            GRN|
|                Asha Worsteling - Greens for Oxley|         Queensland Greens|            GRN|
|                 Mandy Nolan - Greens f

In [23]:
# eyeball false positives: highest-spend matched rows
df.filter(col('political_party').isNotNull() & col('spend_mid').isNotNull() & (col('ad_seq_no') == 1)) \
  .orderBy(col('spend_mid').desc()) \
  .select('page_name', 'bylines', 'political_party', 'spend_mid') \
  .show(20, truncate=60)

+-------------------------------------------------+---------------------------------------------------+---------------+---------+
|                                        page_name|                                            bylines|political_party|spend_mid|
+-------------------------------------------------+---------------------------------------------------+---------------+---------+
|                                  Josh Frydenberg|                                    Josh Frydenberg|             LP|  12499.5|
|                                    Alan Tudge MP|                                      Alan Tudge MP|             LP|   9499.5|
|  Simon Kennedy - Liberal Candidate for Bennelong|Liberal Party of Australia New South Wales Division|             LP|   9499.5|
|                                  Josh Frydenberg|                                    Josh Frydenberg|             LP|   9499.5|
|                                  Josh Frydenberg|                                    Jos

In [25]:
# eyeball false negatives: high-volume political-looking bylines that stayed null

df.filter(col('political_party').isNull() & col('bylines').isNotNull()) \
  .groupBy('bylines').count() \
  .orderBy(desc('count')) \
  .show(30, truncate=80)

+-------------------------------------+------+
|                              bylines| count|
+-------------------------------------+------+
|         Greenpeace Australia Pacific|337937|
|                  Australia for UNHCR| 95987|
|               Australian Labor Party| 66072|
|                Cultural Perspectives| 63872|
|      Amnesty International Australia| 62597|
|   Australian Conservation Foundation| 60698|
|           Liberal Party of Australia| 60558|
|                       Thrive By Five| 59141|
|    Australian Automobile Association| 53113|
|                      Oxfam Australia| 50845|
|                 United Workers Union| 48993|
|      The Pharmacy Guild of Australia| 44735|
|                               Access| 41740|
|                    Australian Unions| 41066|
|        Department of Social Services| 35273|
|                    Advance Australia| 34604|
|               The Wilderness Society| 34076|
|        Victorian Trades Hall Council| 32479|
|            

ok that needs some work.

firstly it looks like we need to add the senate in.

secondly we're missing the acutal parties themselves.

lets check the party add contents

In [27]:
from pyspark.sql.functions import element_at, substring

party_bylines = ['Australian Labor Party', 'Liberal Party of Australia']

for b in party_bylines:
    print(f'\n=== {b} ===')
    df.filter(col('political_party').isNull() & (col('bylines') == b)) \
      .select(
          'page_name',
          'bylines',
          substring(element_at('creative_bodies', 1), 1, 120).alias('body'),
      ) \
      .distinct() \
      .show(10, truncate=120)


=== Australian Labor Party ===


+----------------------+----------------------+------------------------------------------------------------------------------------------------------------------------+
|             page_name|               bylines|                                                                                                                    body|
+----------------------+----------------------+------------------------------------------------------------------------------------------------------------------------+
|Australian Labor Party|Australian Labor Party|Only Anthony Albanese and Labor have a plan for a better future for ALL Australians. \n\nOn May 21, you can vote for ...|
|Australian Labor Party|Australian Labor Party|                   🚨 EXCLUSIVE 🚨 \n\nScott Morrison’s LEAKED plan to tackle the rising cost of living for Australians.|
|             ACT Labor|Australian Labor Party|   ACT needs a strong woman in the Senate. Canberra need Katy’s progressive voice. Vote 1 Labor to elect Katy 

+--------------------------+--------------------------+------------------------------------------------------------------------------------------------------------------------+
|                 page_name|                   bylines|                                                                                                                    body|
+--------------------------+--------------------------+------------------------------------------------------------------------------------------------------------------------+
|Liberal Party of Australia|Liberal Party of Australia|                                                                                    Our plan is working. #StrongerFuture|
|Liberal Party of Australia|Liberal Party of Australia|We want to further help Australians get past the biggest hurdle on their path to home ownership - saving for a deposit -|
|Liberal Party of Australia|Liberal Party of Australia|                                                            

## 9. Write v2 parquet partitioned by party

Drop the intermediate token/term columns before writing — they're large arrays and easy to rebuild. Rows with `political_party = null` land in `political_party=__HIVE_DEFAULT_PARTITION__/` (expected; that's the bulk of the corpus).

In [ ]:
intermediate = ['page_name_safe', 'bylines_safe',
                'page_name_tokens', 'bylines_tokens',
                'page_name_terms', 'bylines_terms']

out = df.drop(*intermediate)

out.write.partitionBy('political_party').parquet(OUT_PATH, mode='overwrite')
print('Wrote:', OUT_PATH)

In [ ]:
# round-trip check
rt = spark.read.parquet(OUT_PATH)
print('Roundtrip rows:', rt.count())
rt.printSchema()